# Grain boundary supercell module: `gbsupercell.py`

In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('../../..'))
if module_path not in sys.path:
    sys.path.append(module_path)
import gbtk.gbsupercell as gbsupercell
import gbtk.grainboundary as grainboundary
import numpy as np

## Creating a grain boundary supercell

So far we have looked at the functionality for specifying a lattice, then a CSL within that lattice and then a grain boundary within that CSL. To actually undertake a calculation, we need to generate a supercell corresponding to a given grain boundary, specifying how many multiples of the minimal grain bounday cell we want to include. We do that now.

A grain boundary supercell contains, of course, a grain boundary (or possibly a pair of equivalent boundaries). Before specifying the details of the supercell, we first need to create the grain boundary. This is done as previously demonstrated:

In [2]:
test_gb = grainboundary.GrainBoundary('fcc')
h = 1; k = 1; l = 1
m = 1; n = 5
test_gb.set_axis([h,k,l])
test_gb.set_angle_mn(m,n)

We will also need the CSL and the DSC lattice to work with, so we calculate these, as follows:

In [3]:
test_gb.set_csl()
test_gb.csl.find_dsc_lattice()

Then, we choose a boundary plane:

In [4]:
test_gb.set_boundary_plane_csl([1,0,0])

True

The next step is to instantiate a supercell object. We pass our grain boundary into the constructor for the supercell:

In [5]:
test_supercell = gbsupercell.Supercell(test_gb)

To obtain useful information, we can switch on the debug output for the supercell (note that we could also separately switch on debug output for the grain boundary, but I have not done that here):

In [6]:
test_supercell.set_debug()

To make it easier to deal with the supercell construction, to make the output easier to understand, and to make writing input files for various pieces of software easier we now rotate the grain boundary cell into a particular orientation with respect to the cartesian axes. This orientation is such that the grain boundary plane lies in the xy-plane and one of the grain boundary cell vectors lies along the x-axis (note that this latter choice, in particular, is helpful when using the output with lammps). In addition, we translate the cell so that it lies predominantly in the positive octant. These transformations are handled by a class method as follows:

In [7]:
test_supercell.calculate_rotation()

----------------------------------------------------
gbsupercell.calculate_rotation() debug:
***************************************
Supercell unit cell vectors (cartesian basis)
[     3.741657    -0.000000    -0.000000 ]
[    -1.870829     3.240370     0.000000 ]
[    -0.000000    -0.000000     1.732051 ]

Basis vectors (cartesian basis)
Black                                       White
[    -0.267261     0.771517    -0.577350 ]  [     0.267261     0.771517    -0.577350 ]
[     0.801784    -0.154303    -0.577350 ]  [     0.534522    -0.617213    -0.577350 ]
[    -0.534522    -0.617213    -0.577350 ]  [    -0.801784    -0.154303    -0.577350 ]



The first part of the debug output shows the rotated supercell vectors in the cartesian basis and you can see how they correspond to the stipulation we laid out above.

Now we begin to specify the supercell. First we give its size, with a list of integers specifying the number of repeats of the grain boundary cell: `[in_plane_dirn_1, in_plane_dirn_2, out_of_plane_black, out_of_plane_white]`. This is done as follows (note that we are also informed of the z-coordinates of the grain boundary planes and the supercell centre):

In [8]:
test_supercell.set_supercell_size(repeats=[1,1,3,3])

----------------------------------------------------
gbsupercell.set_supercell_size() debug:
***************************************
Supercell vectors (cartesian basis)
[     3.741657    -0.000000    -0.000000 ]
[    -1.870829     3.240370     0.000000 ]
[    -0.000000    -0.000000    10.392305 ]
Boundary planes at (z-coordinate)
    2.598076 and     7.794229
Supercell centre at (z-coordinate)
    5.196152



We can now control the microscopic degrees of freedom of the boundary geometry, first by translating the white lattice with respect to the black by some fraction of the DSC lattice vectors:

In [9]:
test_supercell.set_bicrystal_shift(dsc_shift=[0.9,0.9,0.9])

We can also implement a shift in the boundary plane, specified as a fraction of the projection of the DSC lattice diagonal onto the boundary plane normal (the z-direction). This is done as follows:

In [10]:
test_supercell.set_boundary_plane_shift(0.5)

Note that as alternative to specifying the above two adjustments, we can instead specify an in-boundary-plane shift as fractions of the two in-plane CSL vectors. This is done with a command of the following type:

`set_bicrystal_shift(inplane_shift=[0.3,0.2])`

which gives a behaviour suitable for exploring a classic grain boundary gamma surface.

Next we have the option of adding some free volume at each grain boundary. The amount of expansion is specified in length units:

In [11]:
test_supercell.set_expansion(expansion_discrete=0.3)

Thus far we have been assuming that we will work with a periodic supercell with two equivalent grain boundaries. Another option is to have a single boundary in the middle of the cell and free surfaces at the ends. This requires the addition of some vacuum at either end and this can be specified (in length units) as follows:

`test_supercell.set_vacuum(1.0)`

Specifying vacuum in this way will automatically change the type of supercell to one with a single boundary.

Sometimes we may wish to constrain the relaxation of the geometry by fixing blocks of atoms. These blocks will be in the middle of each of the two grains in a supercell containing a pair of boundaries or at the free ends of a supercell containing a single boundary. The width of these blocks in the z-direction is specified in distance units as follows:

In [12]:
test_supercell.set_fix_block(2.0)

The final stage is now to calculate the positions of the atoms and the shape of the supercell, subject to all the adjustments and constraints specified above. This is implemented in a single function, called as follows:

In [13]:
test_supercell.calculate_atom_arrays()

----------------------------------------------------
gbsupercell.calculate_atom_arrays() debug:
******************************************
Filling repeats: Black [(-8 -> 4),(-8 -> 4),(-9 -> 1)]
                 White [(-6 -> 4),(-9 -> 3),(-9 -> 2)]
Original numbers of atoms: Black  504.0, White  504.0
Shifting white crystal by [  0.064,  0.234, -0.106 ]
Shifting boundary planes by -0.059
Numbers of atoms after pareing: Black  252.0, White  252.0
Expanding bicrystal at each boundary by  0.300
Flagging atoms for fixing in place in blocks of depth  2.000

Checking boundary equivalence
Grain boundary eqivalence test: PASS


Note that a final check is run in the case of supercells with a pair of boundaries to make sure that the detailed atomic configuration respects the inversion symmetry of the system (i.e. that the boundaries are equivalent). The result of this test is recorded in the variable `self.boundaries_equivalent` and reported if debug output is switched on.

## Visualising the supercell
The module contains functions for visualising the final supercell. A 3D view is called as follows:

In [14]:
test_supercell.visualise_3d()

And a 2D view, with addtional detail is generated as below:

In [15]:
test_supercell.visualise_2d()

Note that the view above includes a representation of the supercell and boundary planes prior to any shifts of the grain boundary or additions of free volume and vacuum. The various elements of the plot can be hidden by clicking in the legend.

## Writing out the supercell
Now we have our supercell, we want to output it to a suitable form for further study. We can output a lammps input file very simply with the command:

`write_lammps(filename='lammps.txt')`

where we can specify a filename or accept the default `lammps.txt`:

In [18]:
test_supercell.write_lammps(filename='supercells/lammps.txt')

## Some niche features

### Over-filled supercells for illustrative purposes
The main application of the code is expected to be for building grain boundary models for simulation work, so, by default, all the cells respect periodic boundary conditions. However, for some purposes it can be useful to "overfill" the supercell, so that the periodic copies of atoms exacly on the supercell boundary are duplicated on the opposite boundary. This can create a much more natural looking grain boundary model for the purposes of illustrations (or for 3D printing of models).

To request an overfilled model, simply specify the optional flag: `overfill=True` when calling `calculate_atom_arrays()`.

Beware! An over-filled model will create absolute havoc if you load it into a simulation package and specify periodic boundaries!